# Week 13: Vision Transformers & CLIP — Using Pretrained Foundation Models

**This is the final lab.** Last week you *built* a Transformer from its parts and trained
a tiny one. This week we flip it around: instead of building, we **download models other
people already trained on web-scale data** and run them — the way real computer-vision
work actually happens.

One library does all of it — **Hugging Face `transformers`** — and we drive two famous
models with it:

- **ViT** (`google/vit-base-patch16-224`) — image classification, and reading its built-in
  **attention** to see *where it looks*.
- **CLIP** (`openai/clip-vit-base-patch32`) — **zero-shot** classification with labels you
  invent, and **text -> image retrieval**.

Watch for one recurring pattern in every section: **processor -> model -> read the
output.** Once you can picture the *shape* of what each step returns, you can pick up
almost any vision model on the Hub and run it yourself. That is exactly the goal this
whole course has been building toward.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch

%matplotlib inline

### Environment setup

Run this once. On Colab it installs Hugging Face `transformers` and downloads the images
we use. PyTorch is already available on Colab.

The first time you load each model it **downloads the weights** (ViT ~ 330 MB,
CLIP ~ 600 MB) and caches them — later calls are instant. Everything here runs fine on
**CPU**; no GPU needed.

In [ ]:
import os, sys, subprocess, urllib.request

IN_COLAB = "google.colab" in str(get_ipython()) if hasattr(__builtins__, "__IPYTHON__") else False

if IN_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers"])
    REPO_URL = "https://raw.githubusercontent.com/HyeongminLEE/image-processing-tutorial/main"
    os.makedirs("images", exist_ok=True)
    for fname in ["parrots_square.jpg", "mot_color70.jpg", "einstein_monroe.jpg", "calligraphy_hall.jpg"]:
        if not os.path.exists(f"images/{fname}"):
            urllib.request.urlretrieve(f"{REPO_URL}/images/{fname}", f"images/{fname}")
            print(f"Downloaded {fname}")
    IMG_DIR = "images/"
else:
    IMG_DIR = "../images/"

print(f"Running on: {'Google Colab' if IN_COLAB else 'Local'}")
print(f"Image directory: {IMG_DIR}")
print(f"PyTorch version: {torch.__version__}")

### Display helpers

`show_image` draws one image; `show_map` draws one 2-D array; `show_scores` draws one
horizontal bar chart of label -> score (reused for ViT's top-5 and CLIP's probabilities);
`show_attention_overlay` paints an attention heatmap on top of an image. As always, each
helper shows exactly one figure — call it again for another.

In [ ]:
# Each helper shows exactly one figure.

def show_image(img, title=None, scale=4):
    fig, ax = plt.subplots(figsize=(scale, scale))
    if img.ndim == 2:
        ax.imshow(img, cmap="gray", vmin=0, vmax=255)
    else:
        ax.imshow(img)
    if title:
        ax.set_title(title)
    ax.axis("off")
    plt.tight_layout()
    plt.show()


def show_map(m, title=None, scale=4, cmap="gray", colorbar=False):
    fig, ax = plt.subplots(figsize=(scale, scale))
    im = ax.imshow(m, cmap=cmap)
    if colorbar:
        plt.colorbar(im, ax=ax, fraction=0.046)
    if title:
        ax.set_title(title)
    ax.axis("off")
    plt.tight_layout()
    plt.show()


def show_scores(labels, scores, title=None, scale=(6, 3)):
    # labels: list of strings; scores: matching list of floats in [0, 1]
    fig, ax = plt.subplots(figsize=scale)
    positions = range(len(labels))
    ax.barh(positions, scores, color="#3b82f6")
    ax.set_yticks(positions)
    ax.set_yticklabels(labels)
    ax.invert_yaxis()              # first entry on top
    ax.set_xlim(0, 1)
    ax.set_xlabel("score")
    for i in range(len(scores)):
        ax.text(scores[i], i, f" {scores[i]:.2f}", va="center")
    if title:
        ax.set_title(title)
    plt.tight_layout()
    plt.show()


def show_attention_overlay(img, attn, title=None, scale=5):
    # img: H x W (x3) array; attn: a small 2-D map (e.g. 14x14), upsampled for display
    img = np.asarray(img)
    h, w = img.shape[:2]
    fig, ax = plt.subplots(figsize=(scale, scale))
    ax.imshow(img)
    ax.imshow(attn, cmap="jet", alpha=0.5, extent=(0, w, h, 0), interpolation="bilinear")
    if title:
        ax.set_title(title)
    ax.axis("off")
    plt.tight_layout()
    plt.show()

---

## 0. The Hugging Face Hub & `pipeline` — 3 Lines to a Prediction

Last week ended with a text classifier in three lines:

```python
classifier = pipeline("sentiment-analysis")
classifier("This lab finally made attention click!")
```

The **Hugging Face Hub** hosts hundreds of thousands of pretrained models, and `pipeline`
is the highest-level way to run one. The *same three lines* work for images — we just ask
for an image task and point it at a ViT.

In [ ]:
from transformers import pipeline

# Downloads the ViT weights the first time this runs.
classifier = pipeline("image-classification", model="google/vit-base-patch16-224")

In [ ]:
img = Image.open(IMG_DIR + "parrots_square.jpg").convert("RGB")
print("image size (W, H):", img.size)
show_image(np.array(img), title="input image")

In [ ]:
results = classifier(img)

# The output is a list of dicts, each {'label': str, 'score': float}, best first.
print("type:", type(results))
print("number of results:", len(results))
print("results[0]:", results[0])

In [ ]:
for r in results:
    print(f"{r['score']:.3f}  {r['label']}")

Three lines: load the image, build the `pipeline`, call it. But `pipeline` hides three
steps inside — a **processor** that prepares the image, the **model** forward pass, and a
**label lookup**. Next we open that box, because knowing those three pieces is what lets
you drive *any* model on the Hub.

---

## 1. Opening the Box — Processor + Model (ViT)

`pipeline` = **processor -> model -> label lookup**. We now do each step by hand and watch
the shapes. This is the pattern behind every image classification model on the Hub.

In [ ]:
from transformers import AutoImageProcessor

processor = AutoImageProcessor.from_pretrained("google/vit-base-patch16-224")

inputs = processor(images=img, return_tensors="pt")
print("type(inputs):", type(inputs))    # a dict-like BatchFeature
print("keys:", list(inputs.keys()))      # just 'pixel_values'

In [ ]:
pixel_values = inputs["pixel_values"]
print("pixel_values shape:", pixel_values.shape)   # (1, 3, 224, 224) = (batch, channels, H, W)
print("pixel_values dtype:", pixel_values.dtype)   # float32
print("min/max:", pixel_values.min().item(), pixel_values.max().item())

The processor **resized** the image to 224x224, scaled pixels to `[0, 1]`, then
**normalized** by ImageNet mean/std — so the values are no longer 0-255 but roughly
`[-2, 2]`. The model only ever sees this tensor, never the raw image.

In [ ]:
from transformers import AutoModelForImageClassification

# attn_implementation="eager" lets us read attention weights later (Section 2).
model = AutoModelForImageClassification.from_pretrained(
    "google/vit-base-patch16-224", attn_implementation="eager"
)
model.eval()

with torch.no_grad():
    outputs = model(**inputs)

print("type(outputs):", type(outputs))
print("logits shape:", outputs.logits.shape)   # (1, 1000) = one score per ImageNet class

**What are the `Auto...` classes?** They are **dispatchers**, not models. `from_pretrained`
reads the checkpoint's `config.json`, sees `model_type = "vit"`, and hands you the matching
concrete class: `AutoImageProcessor` -> `ViTImageProcessor`, `AutoModelForImageClassification`
-> `ViTForImageClassification`. Every ViT variant shares the **same backbone** (`ViTModel`);
the task class just bolts a different **head** on top — classification here, while
`AutoModel` (no head) or other `AutoModelFor...` heads give you the backbone or a different
task. Swap the checkpoint string and the *identical* code drives a different model — that is
the whole point of `Auto`.

In [ ]:
logits = outputs.logits
pred_id = logits.argmax(dim=-1).item()
print("predicted class id:", pred_id)
print("predicted label:   ", model.config.id2label[pred_id])

# config.id2label maps the 1000 class ids to human-readable names.
print("number of classes:", len(model.config.id2label))
print("id2label[0]:", model.config.id2label[0])

In [ ]:
probs = torch.softmax(logits, dim=-1)[0]
top5 = torch.topk(probs, k=5)

top_labels = []
top_scores = []
for i in range(5):
    class_id = top5.indices[i].item()
    top_labels.append(model.config.id2label[class_id])
    top_scores.append(top5.values[i].item())

show_scores(top_labels, top_scores, title="ViT top-5 (manual pipeline)")

This is exactly what `pipeline` did in Section 0: **processor -> model -> softmax ->
`id2label`**. Three handles — the processor's `pixel_values`, the model's `logits`, and
`config.id2label` — are all you need to run any classification model on the Hub.

---

## 2. ViT Sees With Attention

A CNN needs a *separate* algorithm (Grad-CAM) to reveal where it looked. A ViT carries
that information **inside its forward pass**: the self-attention weights say which patches
each token pulls from. We ask the model to hand them over and look at what the **CLS
token** attends to.

In [ ]:
# Reuse the SAME model from Section 1 (loaded with attn_implementation="eager").
with torch.no_grad():
    outputs = model(**inputs, output_attentions=True)

attentions = outputs.attentions
print("type:", type(attentions))               # a tuple, one entry per encoder layer
print("number of layers:", len(attentions))    # 12
print("one layer shape:", attentions[0].shape) # (1, 12, 197, 197) = (batch, heads, query, key)

197 tokens = **1 CLS token + 196 patches** (a 14x14 grid). For each layer we get a
`(heads, query, key)` block. We want the **last layer**, the **CLS token's row**
(query 0), and only its attention to the **196 patch keys** (columns 1: onward) — averaged
over the 12 heads.

In [ ]:
last_layer = attentions[-1][0]          # (12, 197, 197): drop the batch dim
cls_to_patches = last_layer[:, 0, 1:]   # (12, 196): query = CLS (row 0), keys = patches
cls_to_patches = cls_to_patches.mean(dim=0)   # average over the 12 heads -> (196,)
print("cls_to_patches shape:", cls_to_patches.shape)

attn_map = cls_to_patches.reshape(14, 14)     # the 196 patches back into their grid
print("attn_map shape:", attn_map.shape)

In [ ]:
show_map(attn_map.numpy(), title="CLS attention (14x14 grid)", colorbar=True)
show_attention_overlay(img, attn_map.numpy(), title="Where ViT looks")

Bright = the patches the CLS token draws from when it builds the vector used for
classification. It concentrates on the **birds** and largely ignores the blurred
background — the model's "attention" is literally visible, no extra method required.

---

## 3. CLIP — Zero-Shot Classification

ViT is locked to its **1000 fixed classes** — change the task and you must retrain the
head. **CLIP** removes that ceiling. It was trained to place **images and text in one
shared space**, so you can classify against *any* labels you write — with **no training**.
(This is the contrastive idea from the slides, in action.)

In [ ]:
from transformers import CLIPModel, CLIPProcessor

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()
print("CLIP loaded.")

In [ ]:
labels = ["a parrot", "a dog", "a cat", "an airplane"]

prompts = []
for label in labels:
    prompts.append(f"a photo of {label}.")
print(prompts)

In [ ]:
clip_inputs = clip_processor(text=prompts, images=img, return_tensors="pt", padding=True)
print("keys:", list(clip_inputs.keys()))                          # input_ids, attention_mask, pixel_values
print("input_ids shape:   ", clip_inputs["input_ids"].shape)      # (4, seq_len): 4 prompts
print("pixel_values shape:", clip_inputs["pixel_values"].shape)   # (1, 3, 224, 224): 1 image

The **same processor** tokenizes the text *and* preprocesses the image — both ends of
CLIP. Now one forward pass scores the image against every prompt.

In [ ]:
with torch.no_grad():
    clip_outputs = clip_model(**clip_inputs)

# logits_per_image: similarity of the one image to each of the 4 prompts.
print("logits_per_image shape:", clip_outputs.logits_per_image.shape)   # (1, 4)

probs = clip_outputs.logits_per_image.softmax(dim=-1)[0]
show_scores(labels, probs.tolist(), title="CLIP zero-shot")

CLIP scored the image against each sentence and **softmax** turned the similarities into
probabilities — highest cosine similarity wins. Nothing was trained; swap `labels` for
anything you like and rerun.

### Exercise 3.1 — Classify with labels of your own

Run zero-shot on the street image (`img_street`) with **your own** 4-5 labels. Include
some things that *are* in the scene (a bus, a car, a building) and some that are *not*
(a beach, a forest) so you can see the contrast.

Then try a small **prompt-engineering** test: compare bare labels (`"a bus"`) against the
template (`"a photo of a bus."`) and see whether the scores shift — the slides noted the
template alone adds about +1.3% on ImageNet.

**Hint:** same four steps as the demo — build `prompts`, call `clip_processor(text=...,
images=img_street, ...)`, forward, `softmax`. Reuse `clip_model` and `clip_processor`.

In [ ]:
# Input image (re-loaded so this exercise stands on its own)
img_street = Image.open(IMG_DIR + "mot_color70.jpg").convert("RGB")
show_image(np.array(img_street), title="img_street")

# YOUR CODE HERE
# my_labels = [...]                      # your 4-5 labels
# Build prompts from my_labels, then:
#   clip_processor(text=prompts, images=img_street, return_tensors="pt", padding=True)
#   forward (torch.no_grad) -> logits_per_image -> softmax -> my_probs  (a list)


# --- Output display (uses your my_labels / my_probs) ---
show_scores(my_labels, my_probs, title="CLIP zero-shot (street scene)")


# Did the present labels beat the absent ones? Did the "a photo of ..." template change the
# scores? (You may write your answer in Korean.)
#

---

## 4. CLIP Embeddings & Text -> Image Retrieval

Section 3 used CLIP end-to-end. We can also grab the **vectors** directly:
`get_image_features` and `get_text_features` return points in the shared space. The
**cosine similarity** between a text vector and a bank of image vectors gives us
**retrieval** — type a query, rank the images.

In [ ]:
bank_names = ["parrots_square.jpg", "mot_color70.jpg", "einstein_monroe.jpg", "calligraphy_hall.jpg"]
bank_images = []
for name in bank_names:
    bank_images.append(Image.open(IMG_DIR + name).convert("RGB"))

bank_inputs = clip_processor(images=bank_images, return_tensors="pt")
with torch.no_grad():
    image_features = clip_model.get_image_features(**bank_inputs)
print("image_features shape:", image_features.shape)   # (4, 512): one 512-d vector per image

In [ ]:
query = "a photo of colorful birds"
text_inputs = clip_processor(text=[query], return_tensors="pt", padding=True)
with torch.no_grad():
    text_features = clip_model.get_text_features(**text_inputs)
print("text_features shape:", text_features.shape)   # (1, 512)

# Cosine similarity = dot product of L2-normalized vectors.
image_norm = image_features / image_features.norm(dim=-1, keepdim=True)
text_norm = text_features / text_features.norm(dim=-1, keepdim=True)
similarity = torch.matmul(text_norm, image_norm.T)[0]   # (4,): query vs each image
print("similarity:", similarity)

ranking = torch.argsort(similarity, descending=True)
print("ranking (image index, best first):", ranking.tolist())

In [ ]:
best = ranking[0].item()
show_image(np.array(bank_images[best]), title=f"top match for: '{query}'")

The bird query pulled up the parrots. Change `query` to `"a busy city street"`,
`"a portrait of a person"`, or `"an old building"` and the ranking follows the
**meaning** — not pixels or filenames. That is what CLIP's shared space buys you.

### Exercise 4.1 — Build your own retrieval queries

Reuse the precomputed `image_features` (the 4-image bank). Write **2-3 text queries** of
your own, and for each one retrieve and show the **best-matching** image.

Include one query whose subject **isn't in the bank at all** (e.g. `"a snowy mountain"`)
and see what comes back.

**Hint:** repeat the encode-query -> normalize -> `torch.matmul` -> `argsort` steps for
each query. `image_features` and `image_norm` are already computed above.

**Predict first:** for the *absent* query, will CLIP refuse — or still return its closest
guess?

In [ ]:
# YOUR CODE HERE
# For each of your queries:
#   1. encode the text -> text_features (clip_model.get_text_features)
#   2. L2-normalize, cosine similarity vs image_norm (torch.matmul)
#   3. argsort -> best image index
#   4. show_image(np.array(bank_images[best]), title=query)
# image_features / image_norm from Section 4 are still in memory.


# For the absent query, what did CLIP return — and what does that tell you about similarity
# vs a yes/no decision? (You may write your answer in Korean.)
#

---

### Wrap-up

- One library — **Hugging Face `transformers`** — drove two foundation models: **ViT**
  (classification + built-in attention) and **CLIP** (zero-shot + retrieval).
- The pattern never changed: **processor -> model -> read the output.** Only the output
  differed — `logits` (ViT), `attentions` (where it looks), or image/text **features** (the
  shared space). Inspect the shape, look up the labels or features, done.
- That is the whole course's destination: take an open-source vision model off the Hub,
  understand its inputs and outputs, run inference, and adapt it to your problem — from
  **building** a Transformer by hand last week to **standing on** pretrained giants today.
- **Next week:** what else can a pretrained encoder do beyond classification? Attaching
  task heads for **Detection (YOLO)** and **Segmentation (SAM)**.